# 🧠 EdgeLearningBench — Colab

Notebook per eseguire gli esperimenti del progetto **EdgeLearningBench** su Google Colab.

| Paradigma | Modello | Algoritmo |
|-----------|---------|----------|
| `ff_mlp` | FF-MLP | Forward-Forward greedy |
| `resnet` | ResNet-56 | Backpropagation classica |
| `kan` | KAN | Backpropagation + B-spline |
| `cnn_kan` | CNN-KAN | CNN + KAN Backpropagation |

---
## 1. Caricamento del progetto su Colab

**Opzione A** — Clona da GitHub (decommentare e inserire URL).  
**Opzione B** — Upload manuale di un file `.zip` del progetto.

In [ ]:
import os, sys

PROJECT_DIR = "/content/EdgeLearningBench"

# ── Opzione A: clone da GitHub ──
# REPO_URL = "https://github.com/<tuo-username>/EdgeLearningBench.git"
# !git clone {REPO_URL} {PROJECT_DIR}

# ── Opzione B: upload zip ──
if not os.path.isdir(PROJECT_DIR):
    from google.colab import files
    import zipfile, io
    print("📤 Carica il file .zip del progetto EdgeLearningBench:")
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_name]), 'r') as z:
        z.extractall("/content")
    extracted = [d for d in os.listdir("/content") if os.path.isdir(f"/content/{d}") and d != "sample_data"]
    if len(extracted) == 1 and extracted[0] != "EdgeLearningBench":
        os.rename(f"/content/{extracted[0]}", PROJECT_DIR)
    print(f"✅ Progetto estratto in: {PROJECT_DIR}")
else:
    print(f"✅ Progetto già presente in: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(f"📂 Working dir: {os.getcwd()}")
!ls -la

---
## 2. Installazione dipendenze

In [ ]:
!pip install -q torch torchvision tqdm pyyaml matplotlib

---
## 3. Caricamento configurazione da `config.yaml`

Carica direttamente il file `configs/config.yaml` del progetto.  
Per cambiare paradigma o iperparametri, modifica quel file prima di eseguire.

In [ ]:
import torch
import yaml
from argument import to_namespace
from paths import CHECKPOINT_DIR, DATA_DIR

# Carica il config.yaml del progetto
with open("configs/config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

args = to_namespace(config)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔧 Device: {device}")
if device.type == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
print(f"📋 Paradigma: {args.paradigm}")
print(f"📂 Dataset:   {args.dataset.name}")
print(f"📁 Checkpoint: {CHECKPOINT_DIR}")
print(f"📁 Data:       {DATA_DIR}")

---
## 4. 🚀 Training

Usa la stessa logica di `main.py` per lanciare il training.

In [ ]:
import time
from main import _PARADIGMS
from datasets.dataset import get_dataloaders

# DataLoader
train_loader, test_loader = get_dataloaders(args)
print(f"📊 Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

# Seleziona paradigma
paradigm = args.paradigm
paradigm_args = getattr(args, paradigm)
run_fn = _PARADIGMS[paradigm]

print(f"\n🚀 Avvio training: {paradigm}")
print("═" * 60)

start = time.time()
model = run_fn(paradigm_args, train_loader, test_loader)
elapsed = time.time() - start

print("═" * 60)
print(f"⏱️  Tempo totale: {elapsed / 60:.1f} min ({elapsed:.0f}s)")

---
## 5. 🔍 Predizioni visuali

In [ ]:
import matplotlib.pyplot as plt

model.eval()
dev = next(model.parameters()).device
is_ff = paradigm == "ff_mlp"

images, labels = next(iter(test_loader))
images, labels = images[:16], labels[:16]

with torch.no_grad():
    if is_ff:
        preds = model.predict(images.to(dev)).cpu()
    else:
        preds = model(images.to(dev)).argmax(dim=1).cpu()

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    img = images[i]
    if img.shape[0] == 1:
        ax.imshow(img.squeeze(0), cmap="gray")
    else:
        ax.imshow(img.permute(1, 2, 0).clamp(0, 1))
    color = "green" if preds[i] == labels[i] else "red"
    ax.set_title(f"P:{preds[i].item()} R:{labels[i].item()}", color=color, fontsize=10)
    ax.axis("off")
plt.suptitle(f"{paradigm} — 🟢 Corretto  🔴 Sbagliato", fontsize=13)
plt.tight_layout()
plt.show()

---
## 6. 📈 Riepilogo modello

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"📊 Modello: {paradigm}")
print(f"   Parametri totali: {total_params:,}")
print(f"   Dimensione:       {total_params * 4 / 1e6:.2f} MB (float32)")
print()
print(f"{'Layer':<45} {'Params':>10} {'Shape'}")
print("─" * 80)
for name, p in model.named_parameters():
    print(f"{name:<45} {p.numel():>10,} {list(p.shape)}")

---
## 7. 💾 Download checkpoint

In [ ]:
checkpoint_name = paradigm_args.model.checkpoint_name
checkpoint_path = CHECKPOINT_DIR / checkpoint_name

if checkpoint_path.exists():
    print(f"✅ Checkpoint: {checkpoint_path}")
    print(f"   Dimensione: {checkpoint_path.stat().st_size / 1e6:.2f} MB")
    try:
        from google.colab import files
        files.download(str(checkpoint_path))
        print("📥 Download avviato.")
    except ImportError:
        print("ℹ️  Non su Colab.")
else:
    print(f"⚠️  Checkpoint non trovato: {checkpoint_path}")